In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx
import optax

# Tic-Tac-Toe Reinforcement Learning with JAX and Flax
In this notebook, we build a simple Tic-Tac-Toe environment and train an agent using self-play and policy gradient reinforcement learning.

## 1. Game Environment
First, we define the core mechanics of Tic-Tac-Toe using JAX operations so they can be easily vectorized and compiled.

In [ ]:
def init_game() -> jnp.ndarray:
  return jnp.array([
      0, 0, 0, # Row 1
      0, 0, 0, # Row 2
      0, 0, 0, # Row 3
      0 # Turn
  ])

In [ ]:
def get_valid_moves(board: jnp.ndarray) -> jnp.ndarray:
  return jnp.where(board[:9] == 0, 1, 0)

In [ ]:
def step_game(state: jnp.ndarray, action: int) -> jnp.ndarray:
  turn = state[-1]
  player_mark = (turn % 2) + 1

  return state.at[action].set(player_mark).at[-1].set(turn + 1)

In [ ]:
def check_game(state: jnp.ndarray) -> tuple[jnp.ndarray, jnp.ndarray]:
  # This could probably be optimized further by encoding hard
  # magi values but I thought this was more elegant
  grid = state[:9].reshape((3, 3))
  mapped_grid = jnp.where(grid == 2, -1, grid)

  row_sums = jnp.sum(mapped_grid, axis=1)
  col_sums = jnp.sum(mapped_grid, axis=0)
  diag1_sum = jnp.sum(jnp.diag(mapped_grid))
  diag2_sum = jnp.sum(jnp.diag(jnp.fliplr(mapped_grid)))

  all_sums = jnp.concatenate([row_sums, col_sums, jnp.array([diag1_sum, diag2_sum])])

  p1_won = jnp.any(all_sums == 3)
  p2_won = jnp.any(all_sums == -3)

  is_full = state[-1] >= 9

  winner = jnp.where(p1_won, 1, jnp.where(p2_won, 2, 0))

  is_over = p1_won | p2_won | is_full

  return is_over.astype(jnp.int32), winner.astype(jnp.int32)

In [ ]:
def print_pretty_board(state: jnp.ndarray):
  grid = state[:9].reshape((3, 3))
  turn = state[-1]

  symbols = {0: ".", 1: "X", 2: "O"}

  print("-------------")
  for row in grid:
    row_chars = [symbols[int(cell)] for cell in row]
    print(f"| {row_chars[0]} | {row_chars[1]} | {row_chars[2]} |")
    print("-------------")

  print(f"Turn count: {turn}")

## 2. Neural Network Agent
We define a Multi-Layer Perceptron (MLP) agent using Flax's `nnx` API to evaluate board states and select optimal moves.

In [ ]:
class TicTacToeNet(nnx.Module):
  def __init__(self, hidden_dim: int, *, rngs: nnx.Rngs):
    self.linear1 = nnx.Linear(in_features=9, out_features=hidden_dim, rngs=rngs)
    self.linear2 = nnx.Linear(in_features=hidden_dim, out_features=9, rngs=rngs)

  def __call__(self, state: jnp.ndarray) -> jnp.ndarray:
    board = state[..., :9].astype(jnp.float32)

    x = self.linear1(board)
    x = nnx.relu(x)
    logits = self.linear2(x)
    return logits

In [ ]:
def select_move(logits: jnp.ndarray, state: jnp.ndarray) -> jnp.ndarray:
  valid_moves = get_valid_moves(state)
  valid_logits = jnp.where(valid_moves == 1, logits, jnp.negative(jnp.inf))

  return jnp.argmax(valid_logits)

In [ ]:
def play_one_turn(state: jnp.ndarray, model: TicTacToeNet) -> jnp.ndarray:
  logits = model(state)
  action = select_move(logits, state)

  next_state = step_game(state, action)

  return next_state

In [ ]:
def select_move_stochastic(logits: jnp.ndarray, state: jnp.ndarray, rng: jnp.ndarray) -> jnp.ndarray:
  valid_moves = get_valid_moves(state)

  valid_logits = jnp.where(valid_moves == 1, logits, jnp.negative(jnp.inf))

  # jax.random.categorical handles sampling from unnormalized logits
  action = jax.random.categorical(rng, valid_logits)
  return action

In [ ]:
def play_full_game_stochastic(model: TicTacToeNet, initial_state: jnp.ndarray, rng: jnp.ndarray):
  def game_loop_step_stochastic(state, rng):
    logits = model(state)
    action = select_move_stochastic(logits, state, rng)

    # Get log prob of chosen action
    valid_moves = get_valid_moves(state)
    valid_logits = jnp.where(valid_moves == 1, logits, -jnp.inf)
    log_probs = jax.nn.log_softmax(valid_logits)
    chosen_log_prob = log_probs[action]

    next_state = step_game(state, action)

    is_over, _ = check_game(state)
    next_state = jnp.where(is_over, state, next_state)
    chosen_log_prob = jnp.where(is_over, 0.0, chosen_log_prob)

    return next_state, (state, action, chosen_log_prob)

  # Split the key into 9 unique subkeys for the 9 turns
  turn_keys = jax.random.split(rng, 9)
  final_state, game_history = jax.lax.scan(game_loop_step_stochastic, initial_state, turn_keys)
  return final_state, game_history

In [ ]:
def compute_loss(model: TicTacToeNet, rng: jnp.ndarray):
  # Play a full game using the current model weights
  start_state = init_game()
  final_state, (states, actions, chosen_log_probs) = play_full_game_stochastic(model, start_state, rng)

  _, winner = check_game(final_state)

  # Calculate rewards from the perspective of the active players
  # Player 1 (X) wants winner == 1, Player 2 (O) wants winner == 2
  # We construct a vector of rewards matching who made each move
  turns = states[:, -1]
  player_identities = (turns % 2) + 1 # 1 for P1, 2 for P2

  # Simple reward map: +1 for winning move, -1 for losing move, 0 for draw
  is_win = (winner > 0)
  is_current_player_winner = (winner == player_identities)
  rewards = jnp.where(is_win, jnp.where(is_current_player_winner, 1.0, -1.0), 0.0)

  loss = -jnp.sum(chosen_log_probs * rewards)
  return loss

In [ ]:
@nnx.jit
def train_step(model: TicTacToeNet, optimizer: nnx.Optimizer, train_rng: jnp.ndarray):
  loss_value, grads = nnx.value_and_grad(compute_loss, argnums=0)(model, train_rng)

  optimizer.update(model, grads)

  return loss_value

In [ ]:
batched_loss_fn = jax.vmap(compute_loss, in_axes=(None, 0))

@nnx.jit
def train_step_parallel(model: TicTacToeNet, optimizer: nnx.Optimizer, batch_rngs: jnp.ndarray):
  # A quick wrapper that averages the loss across the whole batch
  def total_loss(m, keys):
    losses = batched_loss_fn(m, keys)
    return jnp.mean(losses)

  loss_value, grads = nnx.value_and_grad(total_loss, argnums=0)(model, batch_rngs)
  optimizer.update(model, grads)
  return loss_value

In [ ]:
# Final Train
master_rng = jax.random.PRNGKey(0)
rngs = nnx.Rngs(42)

model = TicTacToeNet(hidden_dim=64, rngs=rngs)
optimizer = nnx.Optimizer(model, optax.adam(learning_rate=1e-3), wrt=nnx.Param)

print("Starting self-play training...")

BATCH_SIZE = 1024

for epoch in range(20000):
  master_rng, step_rng = jax.random.split(master_rng)
  # Split into 1,024 unique keys for 1,024 parallel games!
  batch_rngs = jax.random.split(step_rng, BATCH_SIZE)

  loss = train_step_parallel(model, optimizer, batch_rngs)
  if epoch % 2000 == 0:
    print(f'({epoch}) Current Loss: {loss}')

Starting self-play training...
(0) Current Loss: 0.5347175598144531
(2000) Current Loss: -0.08946692198514938
(4000) Current Loss: -0.016665760427713394
(6000) Current Loss: -0.009243803098797798
(8000) Current Loss: -0.0
(10000) Current Loss: -0.0
(12000) Current Loss: -0.0
(14000) Current Loss: -0.0
(16000) Current Loss: -0.0
(18000) Current Loss: -0.0


In [ ]:
print("Simulating a game between the model and you:")
state = init_game()
print_pretty_board(state)

while True:
    is_over, winner = check_game(state)
    if is_over:
        if winner == 0:
            print("Game Over! It's a draw.")
        else:
            print(f"Game Over! Player {winner} wins.")
        break

    # Player's Turn
    try:
        move = int(input("Please input a move (0-8): "))
        if not (0 <= move <= 8):
            print("Invalid move! Choose a number between 0 and 8.")
            continue
    except ValueError:
        print("Please enter a valid integer.")
        continue

    state = step_game(state, move)
    print_pretty_board(state)

    # Check if players's move ended the game
    is_over, winner = check_game(state)
    if is_over:
        continue

    # Model's Turn
    print("\nModel is thinking...")
    state = play_one_turn(state, model)
    print_pretty_board(state)

Simulating a game between the model and you:
-------------
| . | . | . |
-------------
| . | . | . |
-------------
| . | . | . |
-------------
Turn count: 0
Please input a move (0-8): 3
-------------
| . | . | . |
-------------
| X | . | . |
-------------
| . | . | . |
-------------
Turn count: 1

Model is thinking...
-------------
| . | . | . |
-------------
| X | O | . |
-------------
| . | . | . |
-------------
Turn count: 2
Please input a move (0-8): 0
-------------
| X | . | . |
-------------
| X | O | . |
-------------
| . | . | . |
-------------
Turn count: 3

Model is thinking...
-------------
| X | . | . |
-------------
| X | O | . |
-------------
| O | . | . |
-------------
Turn count: 4
Please input a move (0-8): 2
-------------
| X | . | X |
-------------
| X | O | . |
-------------
| O | . | . |
-------------
Turn count: 5

Model is thinking...
-------------
| X | O | X |
-------------
| X | O | . |
-------------
| O | . | . |
-------------
Turn count: 6
Please input a mov